# Cross-Encoder Ensemble for Rule Violation Detection

This notebook implements a **cross-encoder ensemble** approach for the Jigsaw community rules hackathon.

**Pipeline:**
1. Fine-tune multiple cross-encoder models (ELECTRA, RoBERTa, DeBERTa) using binary cross-entropy on `(rule, comment)` pairs
2. Augment training data with **cross-rule negatives** — comments that violate a *different* rule serve as hard negatives for the current rule
3. Optionally hold out a stratified validation set to evaluate and stack models
4. Ensemble predictions from multiple models and seeds using a suite of stacking strategies (rank average, optimized weights, logistic regression, etc.)

**Key idea:** Unlike bi-encoder approaches, a cross-encoder jointly attends to both the rule text and the comment in a single forward pass, enabling fine-grained interaction between them.

## Cell 1: Top-level Imports & Environment Setup

Imports standard libraries and sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` to reduce GPU memory fragmentation during training.

In [1]:
import os
import pandas as pd
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import gc
import torch

## Cell 2: `constants.py` — Global Configuration

Writes `constants.py` to disk. Contains all path constants and hyper-parameters:
- `EMBDEDDING_MODEL_PATH`, `DATA_PATH`: Input data and base embedding model paths
- `CROSS_ENCODER_BATCH_SIZE`, `CROSS_ENCODER_EPOCHS`, `CROSS_ENCODER_LEARNING_RATE`: Fine-tuning hyper-parameters (batch=8, 3 epochs, lr=2e-5)
- `CLEAN_TEXT`, `TOP_K`, `BATCH_SIZE`: Legacy constants shared with the embedding pipeline

In [2]:
%%writefile constants.py
#EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
EMBDEDDING_MODEL_PATH = "/kaggle/input/embeddingmodels-repo/downloaded_models/thenlper__gte-large/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# Cross-Encoder specific constants
# CROSS_ENCODER_MODEL_PATH = "/kaggle/input/cross-encoders-v1/downloaded_models/FacebookAI__roberta-base/"  # or "distilroberta-base" for faster training
CROSS_ENCODER_BATCH_SIZE = 8
CROSS_ENCODER_EPOCHS = 3
CROSS_ENCODER_LEARNING_RATE = 2e-5

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Represent this text for semantic similarity search:"
CLEAN_TEXT = True
TOP_K = 50
BATCH_SIZE = 128

Writing constants.py


## Cell 3: `validation_set.py` — Stratified Validation Set Builder

Writes `validation_set.py` to disk. The `prepare_val_set(data_path, n_samples_per_rule, random_state)` function:
1. Builds the same unified corpus as the training pipeline (train split + positive/negative examples embedded in the test set)
2. **Stratified sampling**: extracts `n_samples_per_rule` per rule, split equally between violating and non-violating classes; falls back to `sklearn.train_test_split` with `stratify=` if class sizes are too small
3. Saves the held-out set to `validation.csv`

Keeping these rows out of training prevents data leakage when evaluating and selecting the stacking strategy.

In [3]:
%%writefile validation_set.py

from constants import DATA_PATH
import pandas as pd

def prepare_val_set(DATA_PATH, n_samples_per_rule=10, random_state=42):
    """
    Prepare validation set with stratified sampling based on rule violation.
    
    Parameters:
    -----------
    data_path : str
        Path to the data directory
    n_samples_per_rule : int
        Number of samples to extract per rule (default: 10)
    random_state : int
        Random state for reproducibility (default: 42)
    
    Returns:
    --------
    tuple: (remaining_dataframe, validation_dataframe)
    """
    train_dataset = pd.read_csv(f"{DATA_PATH}/train.csv")
    test_dataset = pd.read_csv(f"{DATA_PATH}/test.csv")
    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    
    dataframe = pd.concat(flatten, axis=0)    
    dataframe = dataframe.drop_duplicates(subset=["body", "rule"], ignore_index=True)
    
    # Extract validation samples with stratified sampling per rule
    validation_samples = []
    #remaining_samples = []

    print("Dataframe CREATED !")
    for rule in dataframe['rule'].unique():
        rule_data = dataframe[dataframe['rule'] == rule]
        
        # Try to get stratified sample with equal violation/non-violation split
        try:
            # Calculate samples per class (violation and non-violation)
            samples_per_class = n_samples_per_rule // 2
            
            violation_data = rule_data[rule_data['rule_violation'] == 1]
            non_violation_data = rule_data[rule_data['rule_violation'] == 0]
            
            # Sample from each class
            if len(violation_data) >= samples_per_class and len(non_violation_data) >= samples_per_class:
                val_violation = violation_data.sample(n=samples_per_class, random_state=random_state)
                val_non_violation = non_violation_data.sample(n=samples_per_class, random_state=random_state)
                
                validation_samples.append(pd.concat([val_violation, val_non_violation]))
                #remaining_samples.append(rule_data.drop(pd.concat([val_violation, val_non_violation]).index))
            else:
                # If not enough samples in either class, use sklearn's stratify
                from sklearn.model_selection import train_test_split
                
                actual_samples = min(n_samples_per_rule, len(rule_data))
                if actual_samples > 0:
                    val_data, rem_data = train_test_split(
                        rule_data, 
                        test_size=len(rule_data) - actual_samples,
                        stratify=rule_data['rule_violation'],
                        random_state=random_state
                    )
                    validation_samples.append(val_data)
                    #remaining_samples.append(rem_data)
                else:
                    print("ERROR1")
                    #remaining_samples.append(rule_data)
                    
        except Exception as e:
            print(f"Warning: Could not stratify for rule '{rule}': {e}")
            # Fallback: simple random sampling
            actual_samples = min(n_samples_per_rule, len(rule_data))
            if actual_samples > 0:
                val_data = rule_data.sample(n=actual_samples, random_state=random_state)
                validation_samples.append(val_data)
                #remaining_samples.append(rule_data.drop(val_data.index))
            else:
                print("ERROR2")
                #remaining_samples.append(rule_data)
    
    # Combine all validation and remaining samples
    validation_df = pd.concat(validation_samples, axis=0).reset_index(drop=True)
    #remaining_df = pd.concat(remaining_samples, axis=0).reset_index(drop=True)
    
    # Save validation set
    validation_df.to_csv("validation.csv", index=False)
    print(f"Validation set saved with {len(validation_df)} samples")
    #print(f"Remaining data: {len(remaining_df)} samples")
    print(f"\nValidation set distribution:")
    print(validation_df.groupby(['rule', 'rule_violation']).size())

Writing validation_set.py


## Cell 4: `utils.py` — Data Loading & Cross-Encoder Utilities

Writes `utils.py` to disk with all helper functions for the cross-encoder pipeline:
- **`build_prompt` / `cleaner`**: Same text formatting and normalization as the embedding pipeline
- **`get_dataframe_to_train(..., include_val, VAL_PATH)`**: Extended version that optionally removes validation rows from the training corpus using a composite `body|||rule` merge key to prevent eval contamination
- **`prepare_dataframe`**: Applies prompting + cleaning; maps labels to `0/1` (not `±1`, since cross-encoders output sigmoid probabilities)
- **`prepare_cross_encoder_data`**: Constructs `(rule_text, comment_text)` sentence pairs and adds **cross-rule negatives** — violating comments from other rules are relabeled as `0` for the current rule, serving as hard negatives
- **`create_cross_encoder_dataset_dict`**: Wraps sentence pairs and labels into a HuggingFace `Dataset` for `CrossEncoderTrainer`

In [4]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist
from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm
from constants import CLEAN_TEXT

def build_prompt(row):
    #return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""
    return f"""{row["body"]}"""

def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )

def get_dataframe_to_train(data_path,include_val,VAL_PATH):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")
    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    dataframe = pd.concat(flatten, axis=0)    
    #dataframe = dataframe.drop_duplicates(ignore_index=True)
    dataframe = dataframe.drop_duplicates(subset=["body", "rule"], ignore_index=True)
    print(f"Original Training data size: {len(dataframe)} samples. Removing Validation Samples...")
    
    # Load validation dataset and remove those rows from training data
    validation_path = "validation.csv"
    try:
        if str(include_val).upper()=="TRUE":
            validation_df = pd.read_csv(validation_path)
            
            # Create a composite key for matching
            dataframe['_merge_key'] = dataframe['body'].astype(str) + '|||' + dataframe['rule'].astype(str)
            validation_df['_merge_key'] = validation_df['body'].astype(str) + '|||' + validation_df['rule'].astype(str)
            
            # Get rows that are NOT in validation set
            initial_count = len(dataframe)
            dataframe = dataframe[~dataframe['_merge_key'].isin(validation_df['_merge_key'])]
            
            # Drop the temporary merge key
            dataframe = dataframe.drop(columns=['_merge_key']).reset_index(drop=True)
            
            removed_count = initial_count - len(dataframe)
            print(f"Removed {removed_count} validation samples from training data")
            print(f"Training data size: {len(dataframe)} samples")
        else:
            print("No Validation Set to be created")
            
    except FileNotFoundError:
        print(f"Warning: validation.csv not found at {validation_path}")
        print("Returning full dataset without filtering")
   
    return dataframe

def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)
    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: 0,
            }
        )
    return dataframe

# Cross-Encoder specific utility functions
def prepare_cross_encoder_data(df, include_examples=True, cross_rule_ratio=0.5,random_seed=42):
    """
    Prepare data for Cross-Encoder training with cross-rule negatives
    Returns sentence pairs and labels
    
    Args:
        df: DataFrame with body, rule, rule_violation
        include_examples: Kept for compatibility (not used)
        cross_rule_ratio: Ratio of cross-rule negatives to add (0.5 = 50% of violations)
    """
    sentence_pairs = []
    labels = []
    
    # Convert rule_violation back to 0/1 format for cross-encoder
    df_copy = df.copy()
    if "rule_violation" in df_copy.columns:
        df_copy["rule_violation"] = df_copy["rule_violation"].map({1: 1, 0: 0})
    
    # 1. Add all original pairs (both violations and non-violations)
    for _, row in df_copy.iterrows():
        if pd.notna(row['body']) and pd.notna(row['rule']):
            # Format: "Rule: {rule_text}" and "Comment: {body}"
            rule_text = f"Rule: {row['rule']}"
            comment_text = f"Comment: {row['body']}"
            
            sentence_pairs.append((rule_text, comment_text))
            labels.append(int(row['rule_violation']))
    
    # 2. Add cross-rule negatives
    # Get all violations
    violations = df_copy[df_copy['rule_violation'] == 1].copy()
    
    if len(violations) > 0:
        # Group violations by rule
        unique_rules = violations['rule'].unique()
        
        cross_rule_count = 0
        
        for rule in unique_rules:
            # Get violations for THIS rule
            this_rule_violations = violations[violations['rule'] == rule]
            
            # Get violations for OTHER rules
            other_rule_violations = violations[violations['rule'] != rule]
            
            if len(other_rule_violations) > 0:
                # Calculate how many cross-rule negatives to add
                n_samples = int(len(this_rule_violations) * cross_rule_ratio)
                n_samples = min(n_samples, len(other_rule_violations))
                
                if n_samples > 0:
                    # Sample violations from other rules
                    sampled = other_rule_violations.sample(n=n_samples, random_state=random_seed, replace=False)
                    
                    for _, violation_row in sampled.iterrows():
                        # This comment violates a different rule, so it's a negative for current rule
                        rule_text = f"Rule: {rule}"
                        comment_text = f"Comment: {violation_row['body']}"
                        
                        sentence_pairs.append((rule_text, comment_text))
                        labels.append(0)
                        cross_rule_count += 1
        
        print(f"Added {cross_rule_count} cross-rule negative examples (seed={random_seed})")

    
    # Print final statistics
    total_pairs = len(labels)
    positive_count = sum(labels)
    negative_count = total_pairs - positive_count
    print(f"Total pairs: {total_pairs}")
    print(f"Positive (violations): {positive_count}")
    print(f"Negative (non-violations): {negative_count}")
    print(f"Positive ratio: {positive_count/total_pairs:.2%}")
    
    return sentence_pairs, labels

def create_cross_encoder_dataset_dict(sentence_pairs, labels):
    """Create Dataset object compatible with CrossEncoderTrainer"""
    from datasets import Dataset
    
    dataset_dict = {
        'sentence1': [pair[0] for pair in sentence_pairs],
        'sentence2': [pair[1] for pair in sentence_pairs], 
        'label': labels
    }
    
    return Dataset.from_dict(dataset_dict)

Writing utils.py


## Cell 5: `finetune_cross_encoder.py` — Cross-Encoder Fine-tuning

Writes the fine-tuning script to disk. The `train_cross_encoder_model(...)` function:
1. **Seeds** Python, NumPy, and PyTorch for reproducibility
2. Loads and cleans the training corpus (excluding validation rows when `include_val=True`)
3. Builds sentence pairs with cross-rule negatives (`cross_rule_ratio=0.4` adds 40% extra hard negatives relative to the number of violations per rule)
4. Trains using `CrossEntropyLoss` for binary classification via `CrossEncoderTrainer`; `save_strategy="no"` skips mid-epoch checkpoints to save disk space
5. Saves the final model to `/kaggle/working/<SAVE_MODEL_PATH>`

Input format fed to the model: `"Rule: <rule_text>"` paired with `"Comment: <comment_body>"`.

In [5]:
%%writefile finetune_cross_encoder.py
import pandas as pd
import logging
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator
from sentence_transformers.cross_encoder.losses.CrossEntropyLoss import CrossEntropyLoss
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments

from utils import get_dataframe_to_train, prepare_dataframe, prepare_cross_encoder_data, create_cross_encoder_dataset_dict, cleaner
from constants import DATA_PATH, CLEAN_TEXT, CROSS_ENCODER_BATCH_SIZE, CROSS_ENCODER_EPOCHS, CROSS_ENCODER_LEARNING_RATE

import os
from tqdm.auto import tqdm

# Disable wandb
os.environ["WANDB_DISABLED"] = "true"

# Set up logging
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)

def train_cross_encoder_model(CROSS_ENCODER_MODEL_PATH, SAVE_MODEL_PATH,include_val,VAL_PATH, cross_rule_ratio=0.4, random_seed=42,):
    """
    Train Cross-Encoder model for rule violation detection
    """

    import random
    import numpy as np
    import torch
    
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_seed)
    
    print(f"Training with random seed: {random_seed}")
    
    print("Loading and preparing data...")
    corpus_dataframe = get_dataframe_to_train(DATA_PATH,include_val,VAL_PATH)
    # Create a list containing the original DataFrame 6 times (1 original + 5 duplicates)
    #corpus_dataframe_list = [corpus_dataframe] * 6
    # Concatenate the list of DataFrames, resetting the index
    #corpus_dataframe = pd.concat(corpus_dataframe_list, ignore_index=True)
    
    if CLEAN_TEXT:
        tqdm.pandas(desc="Cleaning text")
        corpus_dataframe["body"] = corpus_dataframe["body"].progress_apply(cleaner)
    
    print(f"Loaded {len(corpus_dataframe)} samples")
    
    # Prepare cross-encoder data
    sentence_pairs, labels = prepare_cross_encoder_data(corpus_dataframe, include_examples=True,cross_rule_ratio=0.4,random_seed=random_seed)
    print(f"Generated {len(sentence_pairs)} training pairs")
    
    # Check label distribution
    label_counts = pd.Series(labels).value_counts()
    print(f"Label distribution: {label_counts.to_dict()}")
    
    # Split data 
    
    # Create dataset dictionaries
    train_dataset = create_cross_encoder_dataset_dict(sentence_pairs, labels)
    #train_dataset = create_cross_encoder_dataset_dict(train_pairs, train_labels)
    #eval_dataset = create_cross_encoder_dataset_dict(eval_pairs, eval_labels)

    print(f"Training samples: {len(sentence_pairs)}")
    #print(f"Training samples: {len(train_pairs)}")
    #print(f"Evaluation samples: {len(eval_pairs)}")
    
    # Initialize model
    model = CrossEncoder(CROSS_ENCODER_MODEL_PATH, num_labels=2, device="cuda")
    
    # Define loss
    loss = CrossEntropyLoss(model)
    
    # Create evaluator
    
    # Initial evaluation
    #print("Initial model performance:")
    #dev_evaluator(model)
    
    # Training arguments
    output_dir = f"output/reddit-cross-encoder-{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
    short_model_name = CROSS_ENCODER_MODEL_PATH.split("/")[-1]
    run_name = f"reddit-rule-violation-{short_model_name}"
    
    args = CrossEncoderTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=CROSS_ENCODER_EPOCHS,
        per_device_train_batch_size=CROSS_ENCODER_BATCH_SIZE,
        per_device_eval_batch_size=CROSS_ENCODER_BATCH_SIZE,
        warmup_ratio=0.1,
        learning_rate=CROSS_ENCODER_LEARNING_RATE,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",  #这一行加上这个 save_strategy="no"
        logging_steps=1000000,
        seed=random_seed,
        #fp16=True,
        #bf16=False,
        #dataloader_num_workers=4,        # Parallel CPU workers
        #dataloader_pin_memory=True,
        #gradient_accumulation_steps = 2
        
        # Evaluation and saving
        #eval_strategy="steps",
        #eval_steps=200,
        #save_strategy="steps", 
        #save_steps=200,
        #save_total_limit=3,
        #load_best_model_at_end=True,
        #metric_for_best_model="eval_Reddit-Rule-Violation-dev_f1_weighted",
        #greater_is_better=True,
        
        # Logging
        #logging_steps=50,
        #run_name=run_name,
        #report_to=None,
    )
    
    # Create trainer and train
    print("Starting training...")
    trainer = CrossEncoderTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        #eval_dataset=eval_dataset,
        loss=loss,
        #evaluator=dev_evaluator,
    )
    
    trainer.train()
    
    
    # Save the model
    final_output_dir = f"/kaggle/working/{SAVE_MODEL_PATH}"
    model.save_pretrained(final_output_dir)
    print(f"Model saved to: {final_output_dir}")
    
    return model


Writing finetune_cross_encoder.py


## Cell 6: `cross_encoder_inference.py` — Inference & Scoring

Writes the inference script to disk. The `get_cross_encoder_scores(test_dataframe, model_path)` function:
1. Loads the fine-tuned cross-encoder from `/kaggle/working/<model_path>`
2. Constructs `(Rule: ..., Comment: ...)` pairs for every row in the dataframe
3. Runs batch inference (`batch_size=128`) to get raw model outputs
4. **Sigmoid** converts 2-class logits to a violation probability (class 1 score); handles both 2D `(N, 2)` logit outputs and 1D scalar outputs
5. Returns a dataframe with `row_id` and `rule_violation` (probability in `[0, 1]`)

In [6]:
%%writefile cross_encoder_inference.py
import pandas as pd
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm.auto import tqdm
from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, CLEAN_TEXT, CROSS_ENCODER_BATCH_SIZE, CROSS_ENCODER_EPOCHS, CROSS_ENCODER_LEARNING_RATE
import torch.nn.functional as F
import numpy as np
import torch

def sigmoid(x):
    """Apply sigmoid function to convert logits to probabilities"""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def get_cross_encoder_scores(test_dataframe, SAVE_MODEL_PATH):
    """
    Generate cross-encoder predictions for test data
    """
    # Load the trained cross-encoder model
    cross_encoder = CrossEncoder(
        model_name_or_path=f"/kaggle/working/{SAVE_MODEL_PATH}",
        device="cuda",
    )
    
    
    pairs = [(f"Rule: {row['rule']}", f"Comment: {row['prompt']}") 
         for _, row in test_dataframe.iterrows()]

    predictions = cross_encoder.predict(pairs, batch_size=128, convert_to_numpy=True)
    
    # If model outputs logits for 2 classes, convert to probabilities
    if predictions.ndim == 2 and predictions.shape[1] == 2:
        scores = sigmoid(predictions[:, 1])  # Probability of class 1
    else:
        scores = sigmoid(predictions.ravel())
        
    test_dataframe["rule_violation"] = scores
    
    return pd.DataFrame(test_dataframe[["row_id","rule_violation"]])

Writing cross_encoder_inference.py


## Cell 7: `generate_submission.py` — Multi-seed Pipeline Orchestrator

Writes the orchestration script to disk. `generate_submission(model_path, output_file, VAL, output_val_file, seeds)`:
1. Optionally creates a stratified validation set (50 samples/rule via `prepare_val_set`)
2. Loads and preprocesses the test set; when `VAL=TRUE`, concatenates it with the validation set for **simultaneous inference** in a single forward pass per model
3. **For each seed**: trains a fresh cross-encoder, runs inference on the combined dataframe, then splits predictions back into test and validation slices
4. **Averages** raw probabilities across seeds for the final test submission
5. Saves averaged test predictions and per-seed validation predictions (with ground truth) for use by the stacker

CLI args: `<model_path> <output_file> <VAL: TRUE/FALSE> <val_output_file> [seeds: comma-separated]`

In [7]:
%%writefile generate_submission.py
import pandas as pd
import sys 
from constants import DATA_PATH
from utils import get_dataframe_to_train, prepare_dataframe
from finetune_cross_encoder import train_cross_encoder_model
from cross_encoder_inference import get_cross_encoder_scores
from validation_set import prepare_val_set
import torch
import os

def generate_submission(CROSS_ENCODER_MODEL_PATH, OUTPUT_FILE, VAL, OUTPUT_VAL_FILE, seeds=[42, 123, 456]):
    """
    Generate submission using ensemble of models with different seeds.
    Also generates predictions on validation set if it exists.
    """
    import numpy as np

    # Create validation data
    if str(VAL).upper()== "TRUE":
        print("Creating validation set..")
        prepare_val_set(DATA_PATH, n_samples_per_rule=50, random_state=42)
    
    # Load test data
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)
    
    # Check if validation set exists and load it
    validation_path = "validation.csv"
    
    if str(VAL).upper()== "TRUE":
        validation_dataframe = pd.read_csv(validation_path)
        validation_dataframe = prepare_dataframe(validation_dataframe)
        
        # Add a marker to identify validation rows later
        test_dataframe['_is_test'] = True
        validation_dataframe['_is_test'] = False
        
        # Store original indices
        num_test = len(test_dataframe)
        num_validation = len(validation_dataframe)
        
        # Combine test and validation
        combined_dataframe = pd.concat([test_dataframe, validation_dataframe], axis=0, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"Dataset Info:")
        print(f"{'='*60}")
        print(f"Test samples: {num_test}")
        print(f"Validation samples: {num_validation}")
        print(f"Combined samples: {len(combined_dataframe)}")
        print(f"{'='*60}\n")
    else:
        print(f"\nValidation set not found at {validation_path}")
        print("Generating predictions only for test set\n")
        combined_dataframe = test_dataframe.copy()
        num_test = len(test_dataframe)
        num_validation = 0
    
    all_test_predictions = []
    all_validation_predictions = []
    
    # Train and get predictions from each model
    for seed in seeds:
        print(f"\n{'='*60}")
        print(f"Training model with seed {seed}")
        print(f"{'='*60}\n")
        
        SAVE_MODEL_PATH = f"reddit-cross-encoder-model-seed-{seed}"
        
        # Train model with this seed
        train_cross_encoder_model(
            CROSS_ENCODER_MODEL_PATH, 
            SAVE_MODEL_PATH,
            include_val=VAL,
            VAL_PATH=OUTPUT_VAL_FILE,
            cross_rule_ratio=0.4,
            random_seed=seed
            
            
        )
        
        # Get predictions on combined dataset
        predictions = get_cross_encoder_scores(combined_dataframe, SAVE_MODEL_PATH)
        
        # Segregate predictions immediately
        test_preds = predictions.iloc[:num_test]['rule_violation'].values
        #test_preds_ranked = pd.Series(test_preds).rank(method='average') / (len(test_preds) + 1)
        all_test_predictions.append(test_preds)
        print(f"Model {seed} test predictions range: [{test_preds.min():.4f}, {test_preds.max():.4f}]")
    
        
        if str(VAL).upper()== "TRUE":
            val_preds = predictions.iloc[num_test:]['rule_violation'].values
            #val_preds_ranked = pd.Series(val_preds).rank(method='average') / (len(val_preds) + 1)
            all_validation_predictions.append(val_preds)
            print(f"Model {seed} validation predictions range: [{val_preds.min():.4f}, {val_preds.max():.4f}]")
    
    # Average predictions across all seeds
    ensemble_test_predictions = np.mean(all_test_predictions, axis=0)
    
    print(f"\n{'='*60}")
    print(f"Ensemble Statistics (Test):")
    print(f"{'='*60}")
    print(f"Individual model predictions:")
    for i, seed in enumerate(seeds):
        print(f"  Seed {seed}: mean={np.mean(all_test_predictions[i]):.4f}, std={np.std(all_test_predictions[i]):.4f}")
    print(f"Ensemble: mean={np.mean(ensemble_test_predictions):.4f}, std={np.std(ensemble_test_predictions):.4f}")
    print(f"Prediction diversity (std across models): {np.std(all_test_predictions, axis=0).mean():.4f}")
    
    # Create and save test submission
    submission = test_dataframe[["row_id"]].copy()
    submission['rule_violation'] = ensemble_test_predictions
    submission.to_csv(OUTPUT_FILE, index=False)
    print(f"\n{'='*60}")
    print(f"Test submission saved to: {OUTPUT_FILE}")
    print(f"Test predictions: {len(submission)} rows")

    if str(VAL).upper()== "TRUE":
        validation_dataframe['rule_violation_prob'] = val_preds
        validation_dataframe.to_csv(OUTPUT_VAL_FILE,index=False)
        print(f"Validation Set with predictions saved to: {OUTPUT_VAL_FILE}")


if __name__ == "__main__":
    if len(sys.argv) < 3:
        print("Usage: python generate_submission.py <base_model> <output_filename> [seeds]")
        print("Example: python generate_submission.py '/kaggle/input/.../deberta-v3-base' 'submission.csv' 42,123,456")
        sys.exit(1)
    
    CROSS_ENCODER_MODEL_PATH = sys.argv[1]
    OUTPUT_FILE = sys.argv[2]
    VAL = sys.argv[3]
    OUTPUT_VAL_FILE = sys.argv[4]
    
    # Parse seeds if provided
    if len(sys.argv) >= 6:
        seeds = [int(s) for s in sys.argv[5].split(',')]
    else:
        seeds = [42, 123, 456]  # Default seeds
    
    generate_submission(CROSS_ENCODER_MODEL_PATH, OUTPUT_FILE,VAL,OUTPUT_VAL_FILE, seeds)

Writing generate_submission.py


## Cell 8: Run Pipeline — Model 1 (`google/electra-base-discriminator`, seed=42)

Clears GPU memory, then fine-tunes ELECTRA-base as a cross-encoder for 3 epochs with seed 42. Saves test predictions to `submission_m1.csv` and validation predictions (with ground truth labels) to `submission_val_m1.csv`.

In [8]:
gc.collect()
torch.cuda.empty_cache()
! python generate_submission.py "/kaggle/input/cross-encoders-v2/downloaded_models/google__electra-base-discriminator/" "submission_m1.csv" "TRUE" "submission_val_m1.csv" "42"

2025-10-07 17:32:57.726864: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759858378.118853      57 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759858378.230978      57 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Creating validation set..
Dataframe CREATED !
Validation set saved with 100 samples

Validation set distribution:
rule                                                                                                     rule_violation
No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.  0                 25
                                                                             

## Cell 9: Run Pipeline — Model 2 (`FacebookAI/roberta-base`, seed=123)

Clears GPU memory, then fine-tunes RoBERTa-base as a cross-encoder for 3 epochs with seed 123. Saves test predictions to `submission_m2.csv` and validation predictions to `submission_val_m2.csv`.

In [9]:
gc.collect()
torch.cuda.empty_cache()
! python generate_submission.py "/kaggle/input/cross-encoders-v1/downloaded_models/FacebookAI__roberta-base/" "submission_m2.csv" "TRUE" "submission_val_m2.csv" "123"

2025-10-07 17:35:57.905955: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759858557.928645      81 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759858557.935837      81 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Creating validation set..
Dataframe CREATED !
Validation set saved with 100 samples

Validation set distribution:
rule                                                                                                     rule_violation
No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.  0                 25
                                                                             

## Cell 10: Run Pipeline — Model 3 (`microsoft/deberta-v3-base`, seed=456)

Clears GPU memory, then fine-tunes DeBERTa-v3-base as a cross-encoder for 3 epochs with seed 456. Saves test predictions to `submission_m3.csv` and validation predictions to `submission_val_m3.csv`.

In [10]:
gc.collect()
torch.cuda.empty_cache()
! python generate_submission.py "/kaggle/input/huggingfacedebertav3variants/deberta-v3-base" "submission_m3.csv" "TRUE" "submission_val_m3.csv" "456"

2025-10-07 17:38:48.638490: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759858728.660880     105 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759858728.667802     105 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Creating validation set..
Dataframe CREATED !
Validation set saved with 100 samples

Validation set distribution:
rule                                                                                                     rule_violation
No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.  0                 25
                                                                             

## Cell 11: Stacking Ensemble — Select Best Combiner

Evaluates 13 stacking strategies on the held-out validation set (3-fold OOF predictions where applicable) and writes the best-performing one as `submission.csv`.

| Strategy | Description |
|---|---|
| Rank Average | Percentile-rank each model's scores then average — removes absolute scale differences |
| Optimized Weights | `scipy.optimize` finds weights that maximize validation AUC |
| Power Average | Geometric mean of probabilities |
| Simple / Weighted Average | Equal or AUC-proportional linear blend |
| Logistic / Logistic (C=10) / Logistic+Intercept | Linear meta-model variants with different regularization |
| Polynomial Features + Logistic | Adds pairwise interaction terms before logistic regression |
| Ridge Regression | Linear regression meta-model clipped to `[0, 1]` |
| Random Forest / Gradient Boosting | Non-linear tree-based meta-models |
| Neural Network | Small MLP (10→5) trained on the 3 model outputs |

`train_logistic_ensemble` selects the highest-AUC method and saves its test predictions as `submission.csv`; all 13 method-specific files are also generated for reference.

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import PolynomialFeatures
from scipy.optimize import minimize
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

def calculate_correlations(X_val, validation_files):
    """Calculate and visualize correlations between model predictions."""
    print("\n" + "="*60)
    print("Correlation Analysis")
    print("="*60)
    
    # Create correlation matrix
    corr_matrix = np.corrcoef(X_val.T)
    
    # Create a DataFrame for better display
    model_names = [f"Model {i+1}" for i in range(len(validation_files))]
    corr_df = pd.DataFrame(corr_matrix, index=model_names, columns=model_names)
    
    print("\nCorrelation Matrix:")
    print(corr_df.to_string())
    
    # Calculate average correlation (excluding diagonal)
    mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
    avg_correlation = corr_matrix[mask].mean()
    print(f"\nAverage pairwise correlation: {avg_correlation:.4f}")
    
    # Find most and least correlated pairs
    for i in range(len(validation_files)):
        for j in range(i+1, len(validation_files)):
            corr = corr_matrix[i, j]
            print(f"Model {i+1} vs Model {j+1}: {corr:.4f}")
    
    # Diversity score (lower correlation = higher diversity)
    diversity_score = 1 - avg_correlation
    print(f"\nDiversity Score: {diversity_score:.4f} (higher is better)")
    
    return corr_df, avg_correlation


def train_multiple_stackers(X_val, y_val, X_test, validation_files, cv_folds=3):
    """Train multiple stacking models and compare their performance using OOF predictions."""
    print("\n" + "="*60)
    print("Training Multiple Stacking Approaches (with OOF validation)")
    print("="*60)
    print(f"Using {cv_folds}-fold cross-validation for OOF predictions")
    
    stackers = {}
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    
    # Helper function for OOF predictions
    def get_oof_predictions(model, X, y, predict_proba=True):
        oof_preds = np.zeros(len(y))
        for train_idx, val_idx in skf.split(X, y):
            X_train, X_val_fold = X[train_idx], X[val_idx]
            y_train = y[train_idx]
            
            model.fit(X_train, y_train)
            if predict_proba:
                oof_preds[val_idx] = model.predict_proba(X_val_fold)[:, 1]
            else:
                oof_preds[val_idx] = model.predict(X_val_fold)
        return oof_preds
    
    # 1. Rank Average (often better than simple average)
    print("\n1. Rank Average:")
    from scipy.stats import rankdata
    X_val_ranked = np.column_stack([rankdata(X_val[:, i]) / len(X_val) for i in range(X_val.shape[1])])
    X_test_ranked = np.column_stack([rankdata(X_test[:, i]) / len(X_test) for i in range(X_test.shape[1])])
    
    val_pred_rank_avg = X_val_ranked.mean(axis=1)
    test_pred_rank_avg = X_test_ranked.mean(axis=1)
    
    auc_rank_avg = roc_auc_score(y_val, val_pred_rank_avg)
    ap_rank_avg = average_precision_score(y_val, val_pred_rank_avg)
    logloss_rank_avg = log_loss(y_val, val_pred_rank_avg)
    
    print(f"  ROC-AUC: {auc_rank_avg:.4f}")
    print(f"  Average Precision: {ap_rank_avg:.4f}")
    print(f"  Log Loss: {logloss_rank_avg:.4f}")
    
    stackers['Rank Average'] = {
        'model': None,
        'val_pred': val_pred_rank_avg,
        'test_pred': test_pred_rank_avg,
        'auc': auc_rank_avg,
        'ap': ap_rank_avg,
        'logloss': logloss_rank_avg
    }
    
    # 2. Optimized Weighted Average (using scipy.optimize)
    print("\n2. Optimized Weighted Average:")
    
    def objective(weights):
        weights = np.abs(weights)
        weights = weights / weights.sum()
        preds = (X_val * weights).sum(axis=1)
        return -roc_auc_score(y_val, preds)  # Negative because we minimize
    
    initial_weights = np.ones(X_val.shape[1]) / X_val.shape[1]
    result = minimize(objective, initial_weights, method='Nelder-Mead', 
                     options={'maxiter': 10000, 'xatol': 1e-8})
    
    optimal_weights = np.abs(result.x)
    optimal_weights = optimal_weights / optimal_weights.sum()
    
    val_pred_opt_weighted = (X_val * optimal_weights).sum(axis=1)
    test_pred_opt_weighted = (X_test * optimal_weights).sum(axis=1)
    
    auc_opt_weighted = roc_auc_score(y_val, val_pred_opt_weighted)
    ap_opt_weighted = average_precision_score(y_val, val_pred_opt_weighted)
    logloss_opt_weighted = log_loss(y_val, val_pred_opt_weighted)
    
    print(f"  Optimal weights: {optimal_weights}")
    print(f"  ROC-AUC: {auc_opt_weighted:.4f}")
    print(f"  Average Precision: {ap_opt_weighted:.4f}")
    print(f"  Log Loss: {logloss_opt_weighted:.4f}")
    
    stackers['Optimized Weights'] = {
        'model': None,
        'val_pred': val_pred_opt_weighted,
        'test_pred': test_pred_opt_weighted,
        'auc': auc_opt_weighted,
        'ap': ap_opt_weighted,
        'logloss': logloss_opt_weighted,
        'weights': optimal_weights
    }
    
    # 3. Power Average (geometric mean alternative)
    print("\n3. Power Average:")
    epsilon = 1e-10
    X_val_safe = np.clip(X_val, epsilon, 1 - epsilon)
    X_test_safe = np.clip(X_test, epsilon, 1 - epsilon)
    
    val_pred_power = np.exp(np.log(X_val_safe).mean(axis=1))
    test_pred_power = np.exp(np.log(X_test_safe).mean(axis=1))
    
    auc_power = roc_auc_score(y_val, val_pred_power)
    ap_power = average_precision_score(y_val, val_pred_power)
    logloss_power = log_loss(y_val, val_pred_power)
    
    print(f"  ROC-AUC: {auc_power:.4f}")
    print(f"  Average Precision: {ap_power:.4f}")
    print(f"  Log Loss: {logloss_power:.4f}")
    
    stackers['Power Average'] = {
        'model': None,
        'val_pred': val_pred_power,
        'test_pred': test_pred_power,
        'auc': auc_power,
        'ap': ap_power,
        'logloss': logloss_power
    }
    
    # 4. Logistic Regression with higher C (less regularization)
    print("\n4. Logistic Regression (C=10, less regularization):")
    lr_high_c = LogisticRegression(
        fit_intercept=True,
        penalty='l2',
        C=1.0,  # Less regularization
        solver='lbfgs',
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    )
    
    val_pred_lr_high = get_oof_predictions(lr_high_c, X_val, y_val)
    lr_high_c.fit(X_val, y_val)
    test_pred_lr_high = lr_high_c.predict_proba(X_test)[:, 1]
    
    auc_lr_high = roc_auc_score(y_val, val_pred_lr_high)
    ap_lr_high = average_precision_score(y_val, val_pred_lr_high)
    logloss_lr_high = log_loss(y_val, val_pred_lr_high)
    
    print(f"  Coefficients: {lr_high_c.coef_[0]}")
    print(f"  Intercept: {lr_high_c.intercept_[0]:.4f}")
    print(f"  ROC-AUC: {auc_lr_high:.4f}")
    print(f"  Average Precision: {ap_lr_high:.4f}")
    print(f"  Log Loss: {logloss_lr_high:.4f}")
    
    stackers['Logistic (C=10)'] = {
        'model': lr_high_c,
        'val_pred': val_pred_lr_high,
        'test_pred': test_pred_lr_high,
        'auc': auc_lr_high,
        'ap': ap_lr_high,
        'logloss': logloss_lr_high
    }
    
    # 5. Polynomial Features + Logistic Regression
    print("\n5. Polynomial Features (degree=2) + Logistic:")
    poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
    X_val_poly = poly.fit_transform(X_val)
    X_test_poly = poly.transform(X_test)
    
    lr_poly = LogisticRegression(
        fit_intercept=True,
        penalty='l2',
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    )
    
    val_pred_poly = get_oof_predictions(lr_poly, X_val_poly, y_val)
    lr_poly.fit(X_val_poly, y_val)
    test_pred_poly = lr_poly.predict_proba(X_test_poly)[:, 1]
    
    auc_poly = roc_auc_score(y_val, val_pred_poly)
    ap_poly = average_precision_score(y_val, val_pred_poly)
    logloss_poly = log_loss(y_val, val_pred_poly)
    
    print(f"  Number of features: {X_val_poly.shape[1]} (from {X_val.shape[1]})")
    print(f"  ROC-AUC: {auc_poly:.4f}")
    print(f"  Average Precision: {ap_poly:.4f}")
    print(f"  Log Loss: {logloss_poly:.4f}")
    
    stackers['Polynomial Features'] = {
        'model': lr_poly,
        'val_pred': val_pred_poly,
        'test_pred': test_pred_poly,
        'auc': auc_poly,
        'ap': ap_poly,
        'logloss': logloss_poly
    }
    
    # 6. Neural Network Stacker
    print("\n6. Neural Network Stacker:")
    nn = MLPClassifier(
        hidden_layer_sizes=(10, 5),
        activation='relu',
        solver='adam',
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1
    )
    
    val_pred_nn = get_oof_predictions(nn, X_val, y_val)
    nn.fit(X_val, y_val)
    test_pred_nn = nn.predict_proba(X_test)[:, 1]
    
    auc_nn = roc_auc_score(y_val, val_pred_nn)
    ap_nn = average_precision_score(y_val, val_pred_nn)
    logloss_nn = log_loss(y_val, val_pred_nn)
    
    print(f"  ROC-AUC: {auc_nn:.4f}")
    print(f"  Average Precision: {ap_nn:.4f}")
    print(f"  Log Loss: {logloss_nn:.4f}")
    
    stackers['Neural Network'] = {
        'model': nn,
        'val_pred': val_pred_nn,
        'test_pred': test_pred_nn,
        'auc': auc_nn,
        'ap': ap_nn,
        'logloss': logloss_nn
    }
    
    # Keep the original methods too
    # 7. Logistic Regression (Original)
    print("\n7. Logistic Regression (Original):")
    lr = LogisticRegression(
    fit_intercept=False,
    penalty='l2',
    C=0.5,  # reduce C from 0.1 to 0.5 to balance bias/variance on small data
    solver='lbfgs',
    max_iter=500,
    random_state=42,
    class_weight='balanced'
    )
    
    # Get OOF predictions using cross-validation
    val_pred_lr = cross_val_predict(lr, X_val, y_val, cv=cv_folds, method='predict_proba')[:, 1]
    
    # Train on full validation set for test predictions
    lr.fit(X_val, y_val)
    test_pred_lr = lr.predict_proba(X_test)[:, 1]
    
    auc_lr = roc_auc_score(y_val, val_pred_lr)
    ap_lr = average_precision_score(y_val, val_pred_lr)
    logloss_lr = log_loss(y_val, val_pred_lr)
    
    print(f"  Coefficients: {lr.coef_[0]}")
    print(f"  ROC-AUC: {auc_lr:.4f}")
    print(f"  Average Precision: {ap_lr:.4f}")
    print(f"  Log Loss: {logloss_lr:.4f}")
    
    stackers['Logistic Regression'] = {
        'model': lr,
        'val_pred': val_pred_lr,
        'test_pred': test_pred_lr,
        'auc': auc_lr,
        'ap': ap_lr,
        'logloss': logloss_lr
    }
    
    # 8. Logistic Regression with Intercept
    print("\n8. Logistic Regression with Intercept:")
    lr_intercept = LogisticRegression(
    fit_intercept=True,
    penalty='l2',
    C=0.5,   # stronger regularization
    solver='lbfgs',
    max_iter=500,
    random_state=42,
    class_weight='balanced'
    )
    
    # Get OOF predictions
    val_pred_lr_int = cross_val_predict(lr_intercept, X_val, y_val, cv=cv_folds, method='predict_proba')[:, 1]
    
    # Train on full validation set for test predictions
    lr_intercept.fit(X_val, y_val)
    test_pred_lr_int = lr_intercept.predict_proba(X_test)[:, 1]
    
    auc_lr_int = roc_auc_score(y_val, val_pred_lr_int)
    ap_lr_int = average_precision_score(y_val, val_pred_lr_int)
    logloss_lr_int = log_loss(y_val, val_pred_lr_int)
    
    print(f"  Coefficients: {lr_intercept.coef_[0]}")
    print(f"  Intercept: {lr_intercept.intercept_[0]:.4f}")
    print(f"  ROC-AUC: {auc_lr_int:.4f}")
    print(f"  Average Precision: {ap_lr_int:.4f}")
    print(f"  Log Loss: {logloss_lr_int:.4f}")
    
    stackers['Logistic (w/ Intercept)'] = {
        'model': lr_intercept,
        'val_pred': val_pred_lr_int,
        'test_pred': test_pred_lr_int,
        'auc': auc_lr_int,
        'ap': ap_lr_int,
        'logloss': logloss_lr_int
    }
    
    # 9. Ridge Regression
    print("\n9. Ridge Regression Stacker:")
    ridge = Ridge(alpha=2.0, random_state=42)
    
    # Get OOF predictions
    val_pred_ridge = cross_val_predict(ridge, X_val, y_val, cv=cv_folds)
    val_pred_ridge = np.clip(val_pred_ridge, 0, 1)  # Clip to [0,1]
    
    # Train on full validation set for test predictions
    ridge.fit(X_val, y_val)
    test_pred_ridge = ridge.predict(X_test)
    test_pred_ridge = np.clip(test_pred_ridge, 0, 1)
    
    auc_ridge = roc_auc_score(y_val, val_pred_ridge)
    ap_ridge = average_precision_score(y_val, val_pred_ridge)
    logloss_ridge = log_loss(y_val, val_pred_ridge)
    
    print(f"  Coefficients: {ridge.coef_}")
    print(f"  ROC-AUC: {auc_ridge:.4f}")
    print(f"  Average Precision: {ap_ridge:.4f}")
    print(f"  Log Loss: {logloss_ridge:.4f}")
    
    stackers['Ridge Regression'] = {
        'model': ridge,
        'val_pred': val_pred_ridge,
        'test_pred': test_pred_ridge,
        'auc': auc_ridge,
        'ap': ap_ridge,
        'logloss': logloss_ridge
    }
    
    # 10. Random Forest
    print("\n10. Random Forest Stacker:")
    rf = RandomForestClassifier(
    n_estimators=50,     # reduce trees
    max_depth=2,         # shallower trees
    min_samples_leaf=10, # reduce overfitting
    random_state=42,
    n_jobs=-1
    )    
    # Get OOF predictions
    val_pred_rf = cross_val_predict(rf, X_val, y_val, cv=cv_folds, method='predict_proba')[:, 1]
    
    # Train on full validation set for test predictions
    rf.fit(X_val, y_val)
    test_pred_rf = rf.predict_proba(X_test)[:, 1]
    
    auc_rf = roc_auc_score(y_val, val_pred_rf)
    ap_rf = average_precision_score(y_val, val_pred_rf)
    logloss_rf = log_loss(y_val, val_pred_rf)
    
    print(f"  Feature Importances: {rf.feature_importances_}")
    print(f"  ROC-AUC: {auc_rf:.4f}")
    print(f"  Average Precision: {ap_rf:.4f}")
    print(f"  Log Loss: {logloss_rf:.4f}")
    
    stackers['Random Forest'] = {
        'model': rf,
        'val_pred': val_pred_rf,
        'test_pred': test_pred_rf,
        'auc': auc_rf,
        'ap': ap_rf,
        'logloss': logloss_rf
    }
    
    # 11. Gradient Boosting
    print("\n11. Gradient Boosting Stacker:")
    gb = GradientBoostingClassifier(
        n_estimators=30,      # reduce from 50
        max_depth=1,          # weaker base learners
        learning_rate=0.05,   # smaller step
        random_state=42)
    
    # Get OOF predictions
    val_pred_gb = cross_val_predict(gb, X_val, y_val, cv=cv_folds, method='predict_proba')[:, 1]
    
    # Train on full validation set for test predictions
    gb.fit(X_val, y_val)
    test_pred_gb = gb.predict_proba(X_test)[:, 1]
    
    auc_gb = roc_auc_score(y_val, val_pred_gb)
    ap_gb = average_precision_score(y_val, val_pred_gb)
    logloss_gb = log_loss(y_val, val_pred_gb)
    
    print(f"  Feature Importances: {gb.feature_importances_}")
    print(f"  ROC-AUC: {auc_gb:.4f}")
    print(f"  Average Precision: {ap_gb:.4f}")
    print(f"  Log Loss: {logloss_gb:.4f}")
    
    stackers['Gradient Boosting'] = {
        'model': gb,
        'val_pred': val_pred_gb,
        'test_pred': test_pred_gb,
        'auc': auc_gb,
        'ap': ap_gb,
        'logloss': logloss_gb
    }
    
    # 12. Simple Average
    print("\n12. Simple Average (Baseline):")
    val_pred_avg = X_val.mean(axis=1)
    test_pred_avg = X_test.mean(axis=1)
    
    auc_avg = roc_auc_score(y_val, val_pred_avg)
    ap_avg = average_precision_score(y_val, val_pred_avg)
    logloss_avg = log_loss(y_val, val_pred_avg)
    
    print(f"  ROC-AUC: {auc_avg:.4f}")
    print(f"  Average Precision: {ap_avg:.4f}")
    print(f"  Log Loss: {logloss_avg:.4f}")
    
    stackers['Simple Average'] = {
        'model': None,
        'val_pred': val_pred_avg,
        'test_pred': test_pred_avg,
        'auc': auc_avg,
        'ap': ap_avg,
        'logloss': logloss_avg
    }
    
    # 13. Weighted Average (Inverse of validation loss)
    print("\n13. Weighted Average (by performance):")
    individual_aucs = [roc_auc_score(y_val, X_val[:, i]) for i in range(X_val.shape[1])]
    weights = np.array(individual_aucs) / sum(individual_aucs)
    
    val_pred_weighted = (X_val * weights).sum(axis=1)
    test_pred_weighted = (X_test * weights).sum(axis=1)
    
    auc_weighted = roc_auc_score(y_val, val_pred_weighted)
    ap_weighted = average_precision_score(y_val, val_pred_weighted)
    logloss_weighted = log_loss(y_val, val_pred_weighted)
    
    print(f"  Weights: {weights}")
    print(f"  ROC-AUC: {auc_weighted:.4f}")
    print(f"  Average Precision: {ap_weighted:.4f}")
    print(f"  Log Loss: {logloss_weighted:.4f}")
    
    stackers['Weighted Average'] = {
        'model': None,
        'val_pred': val_pred_weighted,
        'test_pred': test_pred_weighted,
        'auc': auc_weighted,
        'ap': ap_weighted,
        'logloss': logloss_weighted,
        'weights': weights
    }
    
    return stackers


def train_logistic_ensemble(validation_files, submission_files, output_file='submission.csv'):
    """
    Train multiple stacking approaches on validation predictions and generate final ensemble.
    """
    
    print("="*60)
    print("Step 1: Loading Validation Files")
    print("="*60)
    
    # Load validation files and extract predictions
    val_predictions = []
    ground_truth = None
    
    for i, val_file in enumerate(validation_files):
        df = pd.read_csv(val_file)
        val_predictions.append(df['rule_violation_prob'].values)
        
        if ground_truth is None:
            ground_truth = df['rule_violation'].values
        
        print(f"Loaded {val_file}: {len(df)} samples")
        print(f"  Mean probability: {df['rule_violation_prob'].mean():.4f}")
        print(f"  Std probability: {df['rule_violation_prob'].std():.4f}")
    
    X_val = np.column_stack(val_predictions)
    y_val = ground_truth
    
    print(f"\nValidation feature matrix shape: {X_val.shape}")
    print(f"Ground truth distribution: {np.bincount(y_val)}")
    print(f"Class balance: {y_val.mean():.4f} positive rate")
    
    # Calculate individual model performance
    print("\n" + "="*60)
    print("Individual Model Performance on Validation:")
    print("="*60)
    
    for i, val_file in enumerate(validation_files):
        auc = roc_auc_score(y_val, X_val[:, i])
        ap = average_precision_score(y_val, X_val[:, i])
        logloss = log_loss(y_val, X_val[:, i])
        print(f"Model {i+1} ({val_file}):")
        print(f"  ROC-AUC: {auc:.4f}")
        print(f"  Average Precision: {ap:.4f}")
        print(f"  Log Loss: {logloss:.4f}")
    
    # Calculate correlations
    corr_df, avg_corr = calculate_correlations(X_val, validation_files)
    
    # Load submission files
    print("\n" + "="*60)
    print("Step 2: Loading Test/Submission Files")
    print("="*60)
    
    test_predictions = []
    row_ids = None
    
    for i, sub_file in enumerate(submission_files):
        df = pd.read_csv(sub_file)
        test_predictions.append(df['rule_violation'].values)
        
        if row_ids is None:
            row_ids = df['row_id'].values
        
        print(f"Loaded {sub_file}: {len(df)} samples")
        print(f"  Mean probability: {df['rule_violation'].mean():.4f}")
        print(f"  Std probability: {df['rule_violation'].std():.4f}")
    
    X_test = np.column_stack(test_predictions)
    print(f"\nTest feature matrix shape: {X_test.shape}")
    
    # Train multiple stackers with OOF validation
    stackers = train_multiple_stackers(X_val, y_val, X_test, validation_files, cv_folds=3)
    
    # Compare all methods
    print("\n" + "="*60)
    print("Comparison of All Stacking Methods:")
    print("="*60)
    
    results_df = pd.DataFrame([
        {
            'Method': name,
            'ROC-AUC': info['auc'],
            'Avg Precision': info['ap'],
            'Log Loss': info['logloss']
        }
        for name, info in stackers.items()
    ])
    results_df = results_df.sort_values('ROC-AUC', ascending=False)
    print(results_df.to_string(index=False))
    
    # Find best method
    best_method = results_df.iloc[0]['Method']
    best_stacker = stackers[best_method]
    
    print(f"\n{'='*60}")
    print(f"Best Method: {best_method}")
    print(f"ROC-AUC: {best_stacker['auc']:.4f}")
    print(f"{'='*60}")
    
    # Generate submissions for all methods
    print("\n" + "="*60)
    print("Step 3: Generating Submissions")
    print("="*60)
    
    for method_name, stacker_info in stackers.items():
        filename = f"submission_{method_name.replace(' ', '_').lower()}.csv"
        
        submission = pd.DataFrame({
            'row_id': row_ids,
            'rule_violation': stacker_info['test_pred']
        })
        
        #submission.to_csv(filename, index=False)
        print(f"Saved: {filename}")
        print(f"  Mean: {stacker_info['test_pred'].mean():.4f}")
        print(f"  Std: {stacker_info['test_pred'].std():.4f}")
    
    # Save best method as main submission
    final_submission = pd.DataFrame({
        'row_id': row_ids,
        'rule_violation': best_stacker['test_pred']
    })
    final_submission.to_csv(output_file, index=False)
    
    print(f"\n{'='*60}")
    print(f"Best submission saved to: {output_file}")
    print(f"Total predictions: {len(final_submission)} rows")
    print("="*60)
    
    return final_submission, stackers, results_df, corr_df


if __name__ == "__main__":
    # Define validation files
    validation_files = [
        'submission_val_m1.csv',
        'submission_val_m2.csv',
        'submission_val_m3.csv'
    ]
    
    # Define submission files
    submission_files = [
        'submission_m1.csv',
        'submission_m2.csv',
        'submission_m3.csv'
    ]
    
    # Train ensemble and generate submissions
    final_submission, stackers, results_df, corr_df = train_logistic_ensemble(
        validation_files=validation_files,
        submission_files=submission_files,
        output_file='submission.csv'
    )
    
    print("\nDone!")
    print("\nGenerated files:")
    print("  - submission.csv (best method)")
    print("  - submission_logistic_regression.csv")
    print("  - submission_logistic_(w/_intercept).csv")
    print("  - submission_ridge_regression.csv")
    print("  - submission_random_forest.csv")
    print("  - submission_gradient_boosting.csv")
    print("  - submission_simple_average.csv")
    print("  - submission_weighted_average.csv")

Step 1: Loading Validation Files
Loaded submission_val_m1.csv: 100 samples
  Mean probability: 0.4945
  Std probability: 0.2014
Loaded submission_val_m2.csv: 100 samples
  Mean probability: 0.4908
  Std probability: 0.2598
Loaded submission_val_m3.csv: 100 samples
  Mean probability: 0.4653
  Std probability: 0.2428

Validation feature matrix shape: (100, 3)
Ground truth distribution: [50 50]
Class balance: 0.5000 positive rate

Individual Model Performance on Validation:
Model 1 (submission_val_m1.csv):
  ROC-AUC: 0.7706
  Average Precision: 0.7879
  Log Loss: 0.5817
Model 2 (submission_val_m2.csv):
  ROC-AUC: 0.7730
  Average Precision: 0.7849
  Log Loss: 0.5727
Model 3 (submission_val_m3.csv):
  ROC-AUC: 0.7522
  Average Precision: 0.7546
  Log Loss: 0.6081

Correlation Analysis

Correlation Matrix:
          Model 1   Model 2   Model 3
Model 1  1.000000  0.733136  0.852998
Model 2  0.733136  1.000000  0.809991
Model 3  0.852998  0.809991  1.000000

Average pairwise correlation: 0.7